Regierungsbezirke map: dissolve Landkreise by first 3 digits of ARS,
show Landkreis outlines (thin) + Regierungsbezirk outlines (thick) + labels.
Same template style as cell 1 (grid, north arrow, scale bar, scientific frame).

In [ ]:

import math
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
from shapely.geometry import box

try:
    import contextily as ctx
    HAS_CTX = True
except Exception:
    HAS_CTX = False

try:
    from matplotlib_map_utils import north_arrow, scale_bar
    HAS_MMU = True
except Exception:
    HAS_MMU = False

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 13,
    "axes.titlesize": 19,
    "axes.labelsize": 13,
    "legend.fontsize": 13,
    "legend.title_fontsize": 14,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
})

# ---------------------------------------------------------------------------
# Paths and config
# ---------------------------------------------------------------------------
PATH_LANDKREISE = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
LAYER_LANDKREISE = "v_vg250_krs"
PATH_NHDA = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Construction_Time_Estimate\NHDA_with_construction_years_RF.gpkg"
OUTPUT_DIR = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen")
OUTPUT_MAP = OUTPUT_DIR / "nhda_centroids_bayern_osm.jpg"

COL_ARS = "Regionalschlüssel_ARS"
BAYERN_PREFIX = "09"
TARGET_CRS = "EPSG:25832"
CRS_NOTE = "CRS: EPSG:25832 - ETRS89 / UTM zone 32N"
GRID_STEP_M = 50000
MAP_PADDING_M = 10000
OSM_ZOOM = 8

POINT_COLOR = "#d90429"
POINT_HALO = "#fff4f6"


def add_matplotlib_grid(ax, bounds, step=GRID_STEP_M):
    minx, miny, maxx, maxy = bounds
    x_start = math.ceil(minx / step) * step
    x_end = math.floor(maxx / step) * step
    y_start = math.ceil(miny / step) * step
    y_end = math.floor(maxy / step) * step

    x_major = [x_start + i * step for i in range(int((x_end - x_start) / step) + 1)] if x_start <= x_end else []
    y_major = [y_start + i * step for i in range(int((y_end - y_start) / step) + 1)] if y_start <= y_end else []

    ax.set_xticks(x_major)
    ax.set_yticks(y_major)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{int(round(x / 1000))}"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f"{int(round(y / 1000))}"))

    if x_major and y_major:
        x_pts, y_pts = [], []
        for xv in x_major:
            for yv in y_major:
                x_pts.append(xv)
                y_pts.append(yv)
        ax.scatter(x_pts, y_pts, marker="+", s=30, linewidths=1.0, color="#8a8a8a", alpha=0.95, zorder=4, clip_on=True)

    ax.tick_params(axis="both", which="major", labelsize=11, length=0, colors="#9B9999")


def add_north_arrow(ax):
    if HAS_MMU:
        north_arrow(
            ax=ax,
            location='upper right',
            size='md',
            rotation={'degrees': 0},
            aob={
                'bbox_to_anchor': (0.985, 0.985),
                'bbox_transform': ax.transAxes,
                'pad': 0.06,
                'borderpad': 0.06,
                'facecolor': 'none',
                'edgecolor': 'none',
                'alpha': 1.0,
                'frameon': False,
            },
        )
        return

    # Manual north arrow: upward-pointing arrow with 'N' above the tip.
    ax.annotate(
        '',
        xy=(0.945, 0.960),      # arrowhead at top
        xytext=(0.945, 0.880),  # tail at bottom
        xycoords='axes fraction',
        textcoords='axes fraction',
        arrowprops=dict(arrowstyle='-|>', color='#222222', linewidth=1.4, shrinkA=0, shrinkB=0),
        zorder=10,
    )
    ax.text(
        0.945, 0.972,
        'N',
        transform=ax.transAxes,
        ha='center', va='bottom',
        fontsize=14, fontweight='bold', color='#222222',
        zorder=10,
    )


def add_scale_bar(ax):
    if HAS_MMU:
        scale_bar(
            ax=ax,
            location="lower right",
            size="xs",
            style="ticks",
            bar={
                "projection": TARGET_CRS,
                "unit": "km",
                "max": 50,
                "major_div": 2,
                "minor_div": 1,
                "minor_type": "none",
                "reverse": False,
            },
            labels={
                "labels": ["0", "25", "50"],
                "style": "major",
                "loc": "below",
                "fontsize": 9,
            },
            units={"loc": "text", "label": "km"},
            text={"fontfamily": "sans-serif", "fontsize": 9, "textcolor": "#222222"},
            aob={
                "pad": 0.0,
                "borderpad": 1.0,
                "facecolor": "none",
                "edgecolor": "none",
                "alpha": 1.0,
                "frameon": False,
            },
        )
        return

    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    span_x = x1 - x0
    span_y = y1 - y0

    total_km = 50
    segment_km = 25
    bar_len = total_km * 1000
    segment_len = segment_km * 1000

    # Position: shifted left (0.45 from right) and slightly higher (0.06 from bottom)
    x_start = x1 - span_x * 0.35
    y_start = y0 + span_y * 0.060

    ax.plot([x_start, x_start + bar_len], [y_start, y_start], color="#222222", linewidth=1.3, zorder=8)
    tick_h = span_y * 0.006
    for x in [x_start, x_start + segment_len, x_start + bar_len]:
        ax.plot([x, x], [y_start - tick_h, y_start + tick_h], color="#222222", linewidth=1.0, zorder=8)

    txt_y = y_start + span_y * 0.011
    ax.text(x_start, txt_y, "0", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)
    ax.text(x_start + segment_len, txt_y, "25", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)
    ax.text(x_start + bar_len, txt_y, "50 km", ha="center", va="bottom", fontsize=9, color="#222222", zorder=8)


def add_scientific_frame(ax, gdf_base):
    ax.set_facecolor("#f1f1f1")
    minx, miny, maxx, maxy = gdf_base.total_bounds
    bounds = (minx - MAP_PADDING_M, miny - MAP_PADDING_M, maxx + MAP_PADDING_M, maxy + MAP_PADDING_M)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.margins(0)
    add_matplotlib_grid(ax, bounds)

    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_autoscale_on(False)

    ax.set_xlabel("Easting (km) - UTM 32N", fontsize=12, color="#555555")
    ax.set_ylabel("Northing (km) - UTM 32N", fontsize=12, color="#555555")
    ax.text(0.01, 0.01, CRS_NOTE, transform=ax.transAxes, ha="left", va="bottom", fontsize=11, color="#555555")

    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
    for side in ["left", "bottom"]:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color("#636262")
        ax.spines[side].set_linewidth(0.8)


def add_legend_bottom_right(ax, n_points):
    handle = Line2D(
        [], [],
        marker="o",
        linestyle="",
        markerfacecolor=POINT_COLOR,
        markeredgecolor="white",
        markeredgewidth=0.7,
        markersize=7,
        label=f"NHDA \ncentroids",
    )
    ax.legend(
        handles=[handle],
        loc="lower left",
        bbox_to_anchor=(1.01, 0.02),
        borderaxespad=0.0,
        frameon=False,
        fontsize=13,
        title_fontsize=14,
        handlelength=1.2,
        labelspacing=0.6,
        borderpad=0.0,
    )

regbez_map = {
    '091': 'Oberbayern',
    '092': 'Niederbayern',
    '093': 'Oberpfalz',
    '094': 'Oberfranken',
    '095': 'Mittelfranken',
    '096': 'Unterfranken',
    '097': 'Schwaben',
}

# Load Landkreise and filter to Bavaria
gdf_lk_rb = gpd.read_file(PATH_LANDKREISE, layer=LAYER_LANDKREISE)
gdf_lk_rb = gdf_lk_rb[gdf_lk_rb[COL_ARS].astype(str).str.startswith(BAYERN_PREFIX)].copy()
gdf_lk_rb = gdf_lk_rb.to_crs(TARGET_CRS)

# Derive Regierungsbezirk code (first 3 digits of ARS)
gdf_lk_rb['regbez_code'] = gdf_lk_rb[COL_ARS].astype(str).str[:3]
gdf_lk_rb['regbez_name'] = gdf_lk_rb['regbez_code'].map(regbez_map)

# Dissolve to Regierungsbezirk boundaries
gdf_rb = gdf_lk_rb.dissolve(by='regbez_name').reset_index()

# Compute centroids for labels
gdf_rb['centroid'] = gdf_rb.geometry.centroid

# ---------------------------------------------------------------------------
# Plot – same template style as cell 1
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8.6, 9.4))
fig.subplots_adjust(right=0.80, left=0.07, top=0.92, bottom=0.08)

# Landkreis fills + thin outlines (background layer)
gdf_lk_rb.plot(ax=ax, facecolor='#f5f5f5', edgecolor='#aaaaaa', linewidth=0.4, zorder=2)

# Regierungsbezirk outlines (thicker, on top)
gdf_rb.plot(ax=ax, facecolor='none', edgecolor='#333333', linewidth=1.5, zorder=3)

# Labels at Regierungsbezirk centroids
for _, row in gdf_rb.iterrows():
    ax.text(
        row['centroid'].x, row['centroid'].y,
        row['regbez_name'],
        ha='center', va='center',
        fontsize=11, fontweight='bold', color='#333333',
        zorder=5,
    )

add_north_arrow(ax)
add_scale_bar(ax)
add_scientific_frame(ax, gdf_lk_rb)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_rb = OUTPUT_DIR / 'regierungsbezirke_bavaria.jpg'
plt.savefig(out_rb, bbox_inches='tight', facecolor='white', format='jpg', dpi=300)
plt.show()
print(f'Saved: {out_rb}')
